# Model 8: Model 7 + Optuna Weight Optimization

**5-fold CV Precision@10:** 0.0562 | **Recall@10:** 0.2951 | **Kaggle score:** 0.1725

Model 8 uses the same architecture as Model 7 (CF + Content + Popularity + Graph RWR) but adds a second CF signal — **count-based CF** (weighted by borrowing frequency) alongside the time-decay CF. All component weights are optimized automatically using **Optuna** with Bayesian search over 200 trials, each evaluated on a 5-fold average.

> **Key finding:** Despite slightly better 5-fold CV score (0.0562 vs 0.0560), the Optuna model scored **0.1725** on Kaggle vs **0.1739** for the hand-tuned Model 7 — showing that 5-fold CV is not a perfect proxy for the true holdout.

## Step 1: Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, diags, bmat
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


## Step 2: Load Data

In [ ]:
df = pd.read_csv('../data/interactions_train.csv').rename(
    columns={'u':'user_id','i':'book_id','t':'timestamp'})
books = pd.read_csv('../data/items.csv')
for c in ['Title','Author','Subjects','Publisher']: books[c] = books[c].fillna('')

uid = {o:n for n,o in enumerate(df['user_id'].unique())}
bid = {o:n for n,o in enumerate(df['book_id'].unique())}
df['user_id'] = df['user_id'].map(uid)
df['book_id']  = df['book_id'].map(bid)
n_users, n_items = df['user_id'].nunique(), df['book_id'].nunique()
print(f'Users: {n_users} | Books: {n_items}')


## Step 3: Helper Functions

Model 8 introduces a **count-based CF matrix** alongside the time-decay matrix from Model 7. Instead of weighting by recency, it weights each (user, book) pair by `log(1 + borrow_count)` — capturing how many times a user borrowed a book, not just when.

In [ ]:
def create_weighted_matrix(data, n_users, n_items, decay=0.03):
    mat = np.zeros((n_users, n_items))
    for _, ud in data.groupby('user_id'):
        ud = ud.sort_values('timestamp'); n = len(ud)
        w = np.array([(1-decay)**(n-1-i) for i in range(n)]); w /= w.max()
        for i, (_, row) in enumerate(ud.iterrows()):
            mat[int(row['user_id']), int(row['book_id'])] = w[i]
    return mat

def create_count_matrix(data, n_users, n_items):
    mat = np.zeros((n_users, n_items))
    for _, row in data.groupby(['user_id','book_id']).size().reset_index(name='c').iterrows():
        mat[int(row['user_id']), int(row['book_id'])] = np.log1p(row['c'])
    return mat

def create_data_matrix(data, n_users, n_items):
    mat = np.zeros((n_users, n_items))
    mat[data['user_id'].values, data['book_id'].values] = 1
    return mat

def ubp(m,s,e=1e-9): return s.dot(m)/(np.abs(s).sum(axis=1)[:,None]+e)
def ibp(m,s,e=1e-9): return (s.dot(m.T)/(s.sum(axis=1)[:,None]+e)).T
def norm(m): lo,hi=m.min(),m.max(); return (m-lo)/(hi-lo+1e-9)
def sln(x): return norm(np.log1p(np.abs(x))*np.sign(x))

def precision_at_k(scores, gt, k=10):
    top_k = np.argpartition(scores,-k,axis=1)[:,-k:]
    return float(gt[np.arange(scores.shape[0])[:,None],top_k].sum(axis=1).mean())/k


## Step 4: Build Content & Graph

In [ ]:
def build_graph(bin_mat, n_users, n_items, alpha=0.7, n_iter=15):
    R=csr_matrix(bin_mat)
    adj=bmat([[csr_matrix((n_users,n_users)),R],[R.T,csr_matrix((n_items,n_items))]],format='csr')
    rs=np.array(adj.sum(axis=1)).flatten(); rs[rs==0]=1
    T=diags(1./rs).dot(adj); nt=n_users+n_items
    sc=np.zeros((n_users,n_items))
    for bs in range(0,n_users,200):
        be=min(bs+200,n_users); p=np.zeros((nt,be-bs))
        for i,u in enumerate(range(bs,be)): p[u,i]=1.
        r=p.copy()
        for _ in range(n_iter): r=alpha*T.dot(r)+(1-alpha)*p
        sc[bs:be]=r[n_users:].T
    return norm(sc)

def build_content(tfidf, train_df, mapping, count_dict, n_users, n_items, bid, books_i):
    profiles=np.zeros((n_users,tfidf.shape[1]),dtype=np.float32)
    ub=train_df.groupby('user_id')['book_id'].apply(list).to_dict()
    for u in range(n_users):
        rows,wts=[],[]
        for b in ub.get(u,[]):
            if b in mapping:
                wts.append(np.log1p(count_dict.get((u,b),1))); rows.append(mapping[b])
        if not rows: continue
        w=np.array(wts,dtype=np.float32); w/=w.sum()
        profiles[u]=np.average(tfidf[rows].toarray(),weights=w,axis=0)
    ns=np.linalg.norm(profiles,axis=1,keepdims=True); ns[ns==0]=1; profiles/=ns
    all_sc=cosine_similarity(profiles,tfidf)
    content=np.zeros((n_users,n_items),dtype=np.float32)
    for i,o in enumerate(books_i):
        if o in bid: content[:,bid[o]]=all_sc[:,i]
    return norm(np.log1p(content)).astype(np.float32)


## Step 5: Precompute All Fold Matrices

To make Optuna fast, we precompute all matrices once before the search begins. Each of the 200 Optuna trials then only needs to combine these matrices with different weights — no recomputation needed.

In [ ]:
books['text']=(books['Title']+' '+books['Author']+' '+books['Author']+' '+
               books['Subjects']+' '+books['Subjects']+' '+books['Publisher'])
tfidf_mat=TfidfVectorizer(max_features=10000,strip_accents='unicode',min_df=2).fit_transform(books['text'])
mapping={bid[o]:i for i,o in enumerate(books['i']) if o in bid}
books_i=books['i'].values

df_s=df.sort_values(['user_id','timestamp']).copy()
df_s['fold']=df_s.groupby('user_id')['timestamp'].transform(
    lambda x: pd.qcut(x.rank(method='first'),5,labels=False))

folds_data = []
for fold in range(5):
    print(f'Fold {fold+1}/5...')
    train_df=df_s[df_s['fold']!=fold]; test_df=df_s[df_s['fold']==fold]
    gt=np.zeros((n_users,n_items),dtype=np.float32)
    gt[test_df['user_id'].values,test_df['book_id'].values]=1
    count_dict={(r['user_id'],r['book_id']):r['c']
                for _,r in train_df.groupby(['user_id','book_id']).size()
                .reset_index(name='c').iterrows()}
    mtx_t=create_weighted_matrix(train_df,n_users,n_items)
    mtx_c=create_count_matrix(train_df,n_users,n_items)
    u_t=sln(ubp(mtx_t,cosine_similarity(mtx_t))).astype(np.float32)
    i_t=sln(ibp(mtx_t,cosine_similarity(mtx_t.T))).astype(np.float32)
    u_c=sln(ubp(mtx_c,cosine_similarity(mtx_c))).astype(np.float32)
    i_c=sln(ibp(mtx_c,cosine_similarity(mtx_c.T))).astype(np.float32)
    content=build_content(tfidf_mat,train_df,mapping,count_dict,n_users,n_items,bid,books_i)
    bin_mat=create_data_matrix(train_df,n_users,n_items)
    pop=norm(np.log1p(bin_mat.sum(axis=0))).astype(np.float32)
    graph=build_graph(bin_mat,n_users,n_items).astype(np.float32)
    folds_data.append({'u_t':u_t,'i_t':i_t,'u_c':u_c,'i_c':i_c,
                       'content':content,'pop':pop,'graph':graph,'gt':gt})
print('All folds ready!')


## Step 6: Optuna Search

Optuna uses **Bayesian optimization** (Tree-structured Parzen Estimator) to search the weight space efficiently. Each trial suggests 7 parameters and is scored on the 5-fold average P@10. After 200 trials, Optuna returns the best combination found.

**Parameters searched:**
- `r_temp` — user vs item ratio in temporal CF
- `r_count` — user vs item ratio in count CF
- `w_cf_temp`, `w_cf_count`, `w_content`, `w_pop`, `w_graph` — component weights

In [ ]:
def objective(trial):
    r_temp   = trial.suggest_float('r_temp',   0.0, 1.0)
    r_count  = trial.suggest_float('r_count',  0.0, 1.0)
    w_cf_temp  = trial.suggest_float('w_cf_temp',  0.0, 1.0)
    w_cf_count = trial.suggest_float('w_cf_count', 0.0, 1.0)
    w_content  = trial.suggest_float('w_content',  0.0, 1.0)
    w_pop      = trial.suggest_float('w_pop',      0.0, 0.2)
    w_graph    = trial.suggest_float('w_graph',    0.0, 1.0)
    scores = []
    for fd in folds_data:
        cf_t = norm(r_temp  * fd['u_t'] + (1-r_temp)  * fd['i_t'])
        cf_c = norm(r_count * fd['u_c'] + (1-r_count) * fd['i_c'])
        hybrid = w_cf_temp*cf_t + w_cf_count*cf_c + w_content*fd['content'] + w_pop*fd['pop'] + w_graph*fd['graph']
        scores.append(precision_at_k(hybrid, fd['gt']))
    return float(np.mean(scores))

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200, show_progress_bar=True)


## Results

In [ ]:
best = study.best_params
print(f'Best 5-fold avg P@10: {study.best_value:.4f}')
print(f'Baseline (Model 7):   0.0560')
print(f'Improvement:         {study.best_value-0.0560:+.4f}')
print()
total_w = sum(v for k,v in best.items() if k.startswith('w_'))
print(f'CF Temporal:  {best["r_temp"]:.2f} user + {1-best["r_temp"]:.2f} item')
print(f'CF Count:     {best["r_count"]:.2f} user + {1-best["r_count"]:.2f} item')
print()
print('Component weights (normalised):')
for k in ['w_cf_temp','w_cf_count','w_content','w_pop','w_graph']:
    print(f'  {k}: {best[k]/total_w:.4f}')


## Key Takeaway

Optuna achieved a slightly higher 5-fold CV score (0.0562) than hand-tuned Model 7 (0.0560), but scored **lower on Kaggle** (0.1725 vs 0.1739). This illustrates a fundamental limitation: the 5-fold CV is an imperfect proxy for the true holdout. Automated search can overfit to the CV metric, finding weights that look better locally but generalise less well to the actual test set.